# Donor Response Classification — notebook corrigido e consolidado

Este notebook substitui a versão anterior por uma pipeline única, sequencial e executável com **vanilla scikit-learn**.

Objetivos:

1. Manter uma história linear: carregamento → auditoria → limpeza conservadora → engenharia de features única → comparação de modelos → seleção final → teste interno → submissão.
2. Corrigir inconsistências de nomes, duplicações e ordem de execução.
3. Evitar leakage: o teste interno é usado apenas uma vez para avaliação final; a escolha do modelo e do threshold é feita em validação.
4. Usar apenas modelos e ferramentas de `scikit-learn`, `pandas` e `numpy`.

## Correções incorporadas

- Removidos imports não usados: `GaussianNB`, `KNeighborsClassifier`, `MLPClassifier` e imports tardios.
- Removidos blocos duplicados de avaliação final.
- Submissão movida para o fim do notebook.
- Substituído qualquer uso ambíguo de `add_donor_features` por uma única função: `add_donor_project_features`.
- Removida a referência incorreta a `lr_grid_expanded`; os resultados ficam centralizados em `model_results`.
- A seleção final deixa de ser fixa e passa a ser baseada no melhor F1 de validação.
- A limpeza foi tornada conservadora: contradições administrativas não são transformadas agressivamente em `NaN`.
- A permutation importance, quando calculada, usa exatamente a matriz de features correspondente ao modelo final.

In [0]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
TARGET = 'TARGET_B'
ID_COL = 'CONTROL_NUMBER'

TRAIN_PATH = Path('donors_train.csv')
TEST_PATH = Path('donors_test.csv')
SUBMISSION_PATH = Path('/mnt/data/submission_corrected_vanilla_sklearn.csv')

## 1. Carregamento dos dados

O dataset de treino contém a variável alvo `TARGET_B`. O ficheiro de teste externo não contém a variável alvo e será usado apenas na etapa final de submissão.

In [0]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print('Train shape:', train_raw.shape)
print('External test shape:', test_raw.shape)
print('Target distribution:')
print(train_raw[TARGET].value_counts(normalize=True).rename('proportion'))

Train shape: (13560, 41)
External test shape: (5812, 40)
Target distribution:
0    0.75
1    0.25
Name: proportion, dtype: float64


## 2. Limpeza conservadora

A limpeza abaixo corrige apenas problemas objetivos:

- placeholders textuais convertidos para `NaN`;
- categorias explicitamente desconhecidas convertidas para `NaN`;
- variáveis numéricas convertidas com `pd.to_numeric`;
- valores negativos em variáveis naturalmente não negativas convertidos para `NaN`;
- percentagens fora de `[0, 100]` convertidas para `NaN`;
- variáveis discretas arredondadas apenas quando estão praticamente inteiras.

Deliberadamente, não removemos sinal por contradições lógicas fracas entre pares de variáveis. Em dados administrativos, essas contradições podem representar ruído de cadastro, atraso de atualização ou codificação histórica; transformá-las agressivamente em `NaN` pode eliminar sinal preditivo útil.

In [0]:
PLACEHOLDERS = ['?', 'NA', 'N/A', '', ' ']
UNKNOWN_CATEGORIES = {
    'DONOR_GENDER': ['U'],
    'HOME_OWNER': ['U'],
    'SES': ['?'],
    'URBANICITY': ['?'],
}

NON_NEGATIVE_COLS = [
    'CHILDREN', 'DONOR_AGE', 'FILE_CARD_GIFT', 'INCOME_GROUP',
    'LIFETIME_CARD_PROM', 'LIFETIME_GIFT_AMOUNT', 'LIFETIME_GIFT_COUNT',
    'LIFETIME_MAX_GIFT_AMT', 'LIFETIME_MIN_GIFT_AMT', 'MONTHS_SINCE_FIRST_GIFT',
    'MONTHS_SINCE_LAST_GIFT', 'MONTHS_SINCE_LAST_PROM_RESP', 'NUMBER_PROM_12',
    'CARD_PROM_12', 'RECENT_CARD_RESPONSE_COUNT', 'RECENT_RESPONSE_COUNT',
    'RECENT_AVG_CARD_GIFT_AMT', 'RECENT_AVG_GIFT_AMT', 'RECENT_CARD_RESPONSE_PROP',
    'RECENT_RESPONSE_PROP', 'LAST_GIFT_AMT', 'PEP_STAR', 'WEALTH_RATING',
    'PER_CAPITA_INCOME', 'LIFETIME_PROM', 'MEDIAN_HOME_VALUE', 'MEDIAN_HOUSEHOLD_INCOME',
]

PERCENT_COLS = [
    'PCT_ATTRIBUTE1', 'PCT_ATTRIBUTE2', 'PCT_ATTRIBUTE3',
    'PCT_ATTRIBUTE4', 'PCT_OWNER_OCCUPIED',
]

DISCRETE_COLS = [
    'CARD_PROM_12', 'CHILDREN', 'DONOR_AGE', 'FILE_CARD_GIFT',
    'FREQUENCY_STATUS_97NK', 'INCOME_GROUP', 'LIFETIME_CARD_PROM',
    'LIFETIME_GIFT_COUNT', 'LIFETIME_PROM', 'MONTHS_SINCE_LAST_GIFT',
    'MONTHS_SINCE_LAST_PROM_RESP', 'NUMBER_PROM_12', 'PEP_STAR',
    'RECENT_CARD_RESPONSE_COUNT', 'RECENT_RESPONSE_COUNT', 'RECENT_STAR_STATUS',
    'WEALTH_RATING',
]


def clean_donors_frame(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy().replace(PLACEHOLDERS, np.nan)

    for col, bad_tokens in UNKNOWN_CATEGORIES.items():
        if col in cleaned.columns:
            cleaned.loc[cleaned[col].isin(bad_tokens), col] = np.nan

    numeric_targets = sorted(set(NON_NEGATIVE_COLS + PERCENT_COLS + DISCRETE_COLS))
    for col in numeric_targets:
        if col in cleaned.columns:
            cleaned[col] = pd.to_numeric(cleaned[col], errors='coerce')

    for col in NON_NEGATIVE_COLS:
        if col in cleaned.columns:
            cleaned.loc[cleaned[col] < 0, col] = np.nan

    for col in PERCENT_COLS:
        if col in cleaned.columns:
            cleaned.loc[(cleaned[col] < 0) | (cleaned[col] > 100), col] = np.nan

    for col in DISCRETE_COLS:
        if col in cleaned.columns:
            near_integer = cleaned[col].notna() & np.isclose(cleaned[col], np.round(cleaned[col]), atol=1e-3)
            fractional = cleaned[col].notna() & ~np.isclose(cleaned[col], np.round(cleaned[col]), atol=1e-3)
            cleaned.loc[near_integer, col] = np.round(cleaned.loc[near_integer, col])
            cleaned.loc[fractional, col] = np.nan

    if ID_COL in cleaned.columns:
        cleaned = cleaned.loc[~cleaned[ID_COL].duplicated(keep='first')].copy()

    return cleaned

## 3. Engenharia de features única

A função abaixo é a única função de feature engineering do projeto. Ela cria razões e flags simples, interpretáveis e compatíveis com modelos lineares e ensembles de árvores.

In [0]:
def add_donor_project_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    def safe_divide(a, b):
        a = pd.to_numeric(a, errors='coerce')
        b = pd.to_numeric(b, errors='coerce')
        return a / b.where(b.notna() & (b != 0), np.nan)

    if {'LIFETIME_GIFT_AMOUNT', 'LIFETIME_GIFT_COUNT'}.issubset(out.columns):
        out['lifetime_avg_gift_amt'] = safe_divide(out['LIFETIME_GIFT_AMOUNT'], out['LIFETIME_GIFT_COUNT'])

    if {'LIFETIME_GIFT_COUNT', 'MONTHS_SINCE_FIRST_GIFT'}.issubset(out.columns):
        out['gift_frequency_per_month'] = safe_divide(out['LIFETIME_GIFT_COUNT'], out['MONTHS_SINCE_FIRST_GIFT'])

    if {'RECENT_AVG_GIFT_AMT', 'lifetime_avg_gift_amt'}.issubset(out.columns):
        out['recent_to_lifetime_avg_gift_ratio'] = safe_divide(out['RECENT_AVG_GIFT_AMT'], out['lifetime_avg_gift_amt'])

    if {'LAST_GIFT_AMT', 'LIFETIME_MAX_GIFT_AMT'}.issubset(out.columns):
        out['last_to_max_gift_ratio'] = safe_divide(out['LAST_GIFT_AMT'], out['LIFETIME_MAX_GIFT_AMT'])

    if {'RECENT_RESPONSE_COUNT', 'NUMBER_PROM_12'}.issubset(out.columns):
        out['recent_response_rate_per_promo'] = safe_divide(out['RECENT_RESPONSE_COUNT'], out['NUMBER_PROM_12'])

    if {'RECENT_CARD_RESPONSE_COUNT', 'CARD_PROM_12'}.issubset(out.columns):
        out['recent_card_response_rate_per_card_promo'] = safe_divide(out['RECENT_CARD_RESPONSE_COUNT'], out['CARD_PROM_12'])

    if {'MONTHS_SINCE_LAST_GIFT', 'MONTHS_SINCE_FIRST_GIFT'}.issubset(out.columns):
        out['recency_to_tenure_ratio'] = safe_divide(out['MONTHS_SINCE_LAST_GIFT'], out['MONTHS_SINCE_FIRST_GIFT'])
        out['donor_active_span_months'] = out['MONTHS_SINCE_FIRST_GIFT'] - out['MONTHS_SINCE_LAST_GIFT']

    if {'NUMBER_PROM_12', 'LIFETIME_PROM'}.issubset(out.columns):
        out['promo_intensity_ratio'] = safe_divide(out['NUMBER_PROM_12'], out['LIFETIME_PROM'])

    for col in ['WEALTH_RATING', 'DONOR_AGE', 'INCOME_GROUP', 'SES', 'URBANICITY', 'HOME_OWNER']:
        if col in out.columns:
            out[f'{col}_missing_flag'] = out[col].isna().astype(int)

    return out.replace([np.inf, -np.inf], np.nan)

## 4. Preparação das matrizes e split honesto

Usamos três partições:

- `train`: ajuste dos modelos;
- `validation`: escolha de modelo e threshold;
- `internal_test`: avaliação final honesta, sem participar da seleção.

In [0]:
train = add_donor_project_features(clean_donors_frame(train_raw))
test_external = add_donor_project_features(clean_donors_frame(test_raw))

X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET].astype(int)
X_external = test_external.drop(columns=[ID_COL])

X_dev, X_internal_test, y_dev, y_internal_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=RANDOM_STATE
)

print('Train:', X_train.shape, y_train.mean())
print('Validation:', X_valid.shape, y_valid.mean())
print('Internal test:', X_internal_test.shape, y_internal_test.mean())

Train: (8136, 54) 0.25
Validation: (2712, 54) 0.25
Internal test: (2712, 54) 0.25


## 5. Pré-processamento comum

O pré-processador é ajustado dentro do `Pipeline`, portanto imputação, escalonamento e one-hot encoding são aprendidos apenas com a partição de treino em cada ajuste.

In [0]:
def make_preprocessor(X_reference: pd.DataFrame) -> ColumnTransformer:
    categorical_cols = X_reference.select_dtypes(include=['object', 'category']).columns.tolist()
    numeric_cols = [col for col in X_reference.columns if col not in categorical_cols]

    try:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse=True)

    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
        ('scaler', StandardScaler()),
    ])

    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', encoder),
    ])

    return ColumnTransformer([
        ('num', numeric_pipe, numeric_cols),
        ('cat', categorical_pipe, categorical_cols),
    ])


def build_pipeline(estimator, X_reference=X_train):
    return Pipeline([
        ('preprocessor', make_preprocessor(X_reference)),
        ('model', clone(estimator)),
    ])

## 6. Funções de threshold tuning e avaliação

Como a métrica de interesse é F1 para a classe positiva, a decisão binária não fica presa ao threshold padrão `0.50`. O threshold é escolhido apenas na validação.

In [0]:
def tune_threshold(y_true, proba):
    # Vectorized enough for notebook execution; avoids repeated sklearn calls in a loop.
    y_arr = np.asarray(y_true).astype(int)
    proba = np.asarray(proba)

    grid = np.unique(np.r_[
        np.linspace(0.02, 0.98, 97),
        np.quantile(proba, np.linspace(0.02, 0.98, 49)),
    ])

    rows = []
    positives = y_arr.sum()
    for threshold in grid:
        pred = (proba >= threshold).astype(int)
        tp = int(((pred == 1) & (y_arr == 1)).sum())
        fp = int(((pred == 1) & (y_arr == 0)).sum())
        fn = int(((pred == 0) & (y_arr == 1)).sum())
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / positives if positives else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append({
            'threshold': float(threshold),
            'f1': f1,
            'precision': precision,
            'recall': recall,
            'prediction_rate': float(pred.mean()),
        })

    return pd.DataFrame(rows).sort_values(
        ['f1', 'precision', 'recall'], ascending=False
    ).reset_index(drop=True)

def evaluate_from_proba(y_true, proba, threshold):
    pred = (proba >= threshold).astype(int)
    return {
        'threshold': float(threshold),
        'f1': f1_score(y_true, pred),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, proba),
        'average_precision': average_precision_score(y_true, proba),
        'prediction_rate': float(pred.mean()),
    }

## 7. Baseline

A baseline “todos positivos” é importante porque a classe positiva representa cerca de 25% da amostra. Nesse cenário, prever todos como positivo gera F1 próximo de 0.40. Um modelo útil deve superar essa referência de forma estável.

In [0]:
baseline_pred = np.ones_like(y_valid)
baseline_f1 = f1_score(y_valid, baseline_pred)
print(f'Validation F1 — all-positive baseline: {baseline_f1:.4f}')

Validation F1 — all-positive baseline: 0.4000


## 8. Comparação de modelos finalistas

Para preservar clareza, a versão final compara apenas quatro famílias declaradas desde o início:

- Logistic Regression;
- Decision Tree;
- Random Forest;
- Extra Trees.

Modelos avançados, stacking, calibração e undersampling foram removidos desta versão consolidada porque não melhoraram de forma robusta a narrativa nem a avaliação honesta.

In [0]:
candidate_specs = [
    {
        'name': 'Logistic Regression',
        'estimator': LogisticRegression(solver='liblinear', max_iter=1000, random_state=RANDOM_STATE),
        'params': {
            'model__C': [0.003, 0.03, 0.1],
            'model__class_weight': ['balanced', None],
        },
    },
    {
        'name': 'Decision Tree',
        'estimator': DecisionTreeClassifier(random_state=RANDOM_STATE),
        'params': {
            #'criterion' : ['gini'],
            'model__max_depth': [3, 5],
            'model__min_samples_leaf': [50],
            'model__class_weight': ['balanced'],
        },
    },
    {
        'name': 'Random Forest',
        'estimator': RandomForestClassifier(n_estimators=60, n_jobs=1, random_state=RANDOM_STATE),
        'params': {
            #'criterion': ['gini'],
            'model__max_depth': [5],
            'model__min_samples_leaf': [20],
            'model__class_weight': ['balanced_subsample'],
        },
    },
    {
        'name': 'Extra Trees',
        'estimator': ExtraTreesClassifier(n_estimators=60, n_jobs=1, random_state=RANDOM_STATE),
        'params': {
            'model__max_depth': [5],
            'model__min_samples_leaf': [20],
            'model__class_weight': ['balanced'],
        },
    },
]

model_rows = []
fitted_candidates = {}

for spec in candidate_specs:
    for params in ParameterGrid(spec['params']):
        pipe = build_pipeline(spec['estimator'])
        pipe.set_params(**params)
        pipe.fit(X_train, y_train)

        valid_proba = pipe.predict_proba(X_valid)[:, 1]
        threshold_table = tune_threshold(y_valid, valid_proba)
        best_threshold = float(threshold_table.loc[0, 'threshold'])
        metrics = evaluate_from_proba(y_valid, valid_proba, best_threshold)

        model_key = f"{spec['name']} | {params}"
        fitted_candidates[model_key] = {
            'pipeline': pipe,
            'threshold': best_threshold,
            'valid_proba': valid_proba,
            'params': params,
            'family': spec['name'],
            'estimator': spec['estimator'],
        }

        model_rows.append({
            'model_key': model_key,
            'family': spec['name'],
            **params,
            **metrics,
        })

model_results = pd.DataFrame(model_rows).sort_values(
    ['f1', 'precision', 'recall'], ascending=False
).reset_index(drop=True)

model_results.head(15)

Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🏃 View run languid-calf-362 at: https://dbc-701a66cb-661b.cloud.databricks.com/ml/experiments/2760995407200007/runs/7dd4756d1a46481d939aa1e0a961eaa7
🧪 View experiment at: https://dbc-701a66cb-661b.cloud.databricks.com/ml/experiments/2760995407200007


Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

...

🏃 View run bright-mink-354 at: https://dbc-701a66cb-661b.cloud.databricks.com/ml/experiments/2760995407200007/runs/470d63e16c23407f9e21d434ad5e9afd
🧪 View experiment at: https://dbc-701a66cb-661b.cloud.databricks.com/ml/experiments/2760995407200007


Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🏃 View run sedate-doe-384 at: https://dbc-701a66cb-661b.cloud.databricks.com/ml/experiments/2760995407200007/runs/72aa1d5d7aad48b78b7dcf7c6f703638
🧪 View experiment at: https://dbc-701a66cb-661b.cloud.databricks.com/ml/experiments/2760995407200007


,model_key,family,model__C,model__class_weight,threshold,f1,precision,recall,roc_auc,average_precision,prediction_rate,model__max_depth,model__min_samples_leaf
0,Extra Trees | {'model__class_weight': 'balance...,Extra Trees,NaN,balanced,0.481563,0.421102,0.308060,0.665192,0.616531,0.357333,0.539823,5.0,20.0
1,Random Forest | {'model__class_weight': 'balan...,Random Forest,NaN,balanced_subsample,0.460000,0.420237,0.304090,0.679941,0.614898,0.377250,0.558997,5.0,20.0
2,"Logistic Regression | {'model__C': 0.003, 'mod...",Logistic Regression,0.003,balanced,0.420000,0.416795,0.282065,0.797935,0.611761,0.340965,0.707227,NaN,NaN
3,"Logistic Regression | {'model__C': 0.1, 'model...",Logistic Regression,0.100,None,0.158403,0.416779,0.270413,0.908555,0.604036,0.339057,0.839971,NaN,NaN
4,"Logistic Regression | {'model__C': 0.1, 'model...",Logistic Regression,0.100,balanced,0.370000,0.416667,0.272480,0.884956,0.604159,0.337961,0.811947,NaN,NaN
5,"Logistic Regression | {'model__C': 0.03, 'mode...",Logistic Regression,0.030,balanced,0.394057,0.416210,0.276565,0.840708,0.605643,0.338997,0.759956,NaN,NaN
6,"Logistic Regression | {'model__C': 0.03, 'mode...",Logistic Regression,0.030,None,0.170000,0.415557,0.272518,0.874631,0.605427,0.339848,0.802360,NaN,NaN
7,"Logistic Regression | {'model__C': 0.003, 'mod...",Logistic Regression,0.003,None,0.216854,0.414911,0.286034,0.755162,0.609011,0.339671,0.660029,NaN,NaN
8,Decision Tree | {'model__class_weight': 'balan...,Decision Tree,NaN,balanced,0.380000,0.411517,0.270046,0.864307,0.603851,0.333759,0.800147,5.0,50.0
9,Decision Tree | {'model__class_weight': 'balan...,Decision Tree,NaN,balanced,0.360000,0.400275,0.260850,0.859882,0.582671,0.306093,0.824115,3.0,50.0


## 9. Seleção final baseada em evidência

A seleção abaixo é automática: escolhe o maior F1 de validação. A meta de F1 0.45–0.51 é tratada como objetivo, não como promessa. Se nenhum modelo atingir a faixa em validação, o notebook declara isso explicitamente e seleciona a melhor alternativa honesta.

In [0]:
TARGET_F1_MIN = 0.45
TARGET_F1_MAX = 0.51

models_in_target_band = model_results.query('@TARGET_F1_MIN <= f1 <= @TARGET_F1_MAX').copy()

# Tie-breaker conservador: quando os modelos estão praticamente empatados em F1
# de validação, preferimos a família mais simples e estável. Isto evita selecionar
# um ensemble marginalmente melhor por ruído de validação.
complexity_rank = {
    'Logistic Regression': 1,
    'Decision Tree': 2,
    'Random Forest': 3,
    'Extra Trees': 4,
}
MODEL_SELECTION_TOLERANCE = 0.005

def choose_with_simplicity_tiebreaker(results_df):
    best_f1 = results_df['f1'].max()
    candidate_pool = results_df[results_df['f1'] >= best_f1 - MODEL_SELECTION_TOLERANCE].copy()
    candidate_pool['complexity_rank'] = candidate_pool['family'].map(complexity_rank)
    return candidate_pool.sort_values(
        ['complexity_rank', 'f1', 'precision', 'recall'],
        ascending=[True, False, False, False]
    ).iloc[0]

if len(models_in_target_band) > 0:
    selected_row = choose_with_simplicity_tiebreaker(models_in_target_band)
    selection_reason = 'Modelo selecionado dentro da faixa alvo de F1 na validação, com desempate por simplicidade.'
else:
    selected_row = choose_with_simplicity_tiebreaker(model_results)
    selection_reason = (
        'Nenhum modelo atingiu F1 entre 0.45 e 0.51 na validação. '
        'Selecionado o melhor modelo honesto de validação, usando desempate conservador por simplicidade quando a diferença de F1 é marginal.'
    )

best_model_key = selected_row['model_key']
best_candidate = fitted_candidates[best_model_key]
best_model = best_candidate['pipeline']
selected_threshold = float(best_candidate['threshold'])

print(selection_reason)
print('Selected model:', best_model_key)
print(f'Selected validation threshold: {selected_threshold:.4f}')
print('\nValidation metrics:')
print(selected_row[['f1', 'precision', 'recall', 'roc_auc', 'average_precision', 'prediction_rate']])

Nenhum modelo atingiu F1 entre 0.45 e 0.51 na validação. Selecionado o melhor modelo honesto de validação, usando desempate conservador por simplicidade quando a diferença de F1 é marginal.
Selected model: Logistic Regression | {'model__C': 0.003, 'model__class_weight': 'balanced'}
Selected validation threshold: 0.4200

Validation metrics:
f1                   0.416795
precision            0.282065
recall               0.797935
roc_auc              0.611761
average_precision    0.340965
prediction_rate      0.707227
Name: 2, dtype: object


## 10. Avaliação final no teste interno

Esta é a única avaliação no teste interno. O threshold vem da validação; portanto, o teste interno não participa do tuning.

In [0]:
internal_proba = best_model.predict_proba(X_internal_test)[:, 1]
internal_metrics = evaluate_from_proba(y_internal_test, internal_proba, selected_threshold)
internal_pred = (internal_proba >= selected_threshold).astype(int)

print('Internal test metrics:')
print(pd.Series(internal_metrics).round(4))
print('\nConfusion matrix:')
print(confusion_matrix(y_internal_test, internal_pred))
print('\nClassification report:')
print(classification_report(y_internal_test, internal_pred, digits=4))

Internal test metrics:
threshold            0.4200
f1                   0.4055
precision            0.2732
recall               0.7861
roc_auc              0.5927
average_precision    0.3238
prediction_rate      0.7194
dtype: float64

Confusion matrix:
[[ 616 1418]
 [ 145  533]]

Classification report:
              precision    recall  f1-score   support

           0     0.8095    0.3029    0.4408      2034
           1     0.2732    0.7861    0.4055       678

    accuracy                         0.4237      2712
   macro avg     0.5413    0.5445    0.4231      2712
weighted avg     0.6754    0.4237    0.4320      2712



## 11. Permutation importance coerente com o modelo final

A permutation importance é calculada sobre `X_internal_test`, que contém exatamente as mesmas features engineered usadas pelo pipeline finalista. A métrica usa o threshold selecionado na validação.

In [0]:
RUN_PERMUTATION_IMPORTANCE = False

if RUN_PERMUTATION_IMPORTANCE:
    def threshold_f1_scorer(estimator, X_eval, y_eval):
        proba = estimator.predict_proba(X_eval)[:, 1]
        pred = (proba >= selected_threshold).astype(int)
        return f1_score(y_eval, pred)

    importance_sample_size = min(500, len(X_internal_test))
    X_importance = X_internal_test.sample(importance_sample_size, random_state=RANDOM_STATE)
    y_importance = y_internal_test.loc[X_importance.index]

    perm = permutation_importance(
        best_model,
        X_importance,
        y_importance,
        scoring=threshold_f1_scorer,
        n_repeats=2,
        random_state=RANDOM_STATE,
        n_jobs=1,
    )

    importance_table = pd.DataFrame({
        'feature': X_importance.columns,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std,
    }).sort_values('importance_mean', ascending=False).reset_index(drop=True)
else:
    importance_table = pd.DataFrame({
        'feature': X_internal_test.columns,
        'importance_mean': np.nan,
        'importance_std': np.nan,
    })
    print('Permutation importance desativada por padrão para manter o notebook leve.')
    print('Para executar, altere RUN_PERMUTATION_IMPORTANCE = True nesta célula.')

importance_table.head(15)

Permutation importance desativada por padrão para manter o notebook leve.
Para executar, altere RUN_PERMUTATION_IMPORTANCE = True nesta célula.


,feature,importance_mean,importance_std
0,CARD_PROM_12,NaN,NaN
1,CHILDREN,NaN,NaN
2,DONOR_AGE,NaN,NaN
3,DONOR_GENDER,NaN,NaN
4,FILE_CARD_GIFT,NaN,NaN
5,FREQUENCY_STATUS_97NK,NaN,NaN
6,HOME_OWNER,NaN,NaN
7,INCOME_GROUP,NaN,NaN
8,LAST_GIFT_AMT,NaN,NaN
9,LIFETIME_CARD_PROM,NaN,NaN


## 12. Refit final e submissão

Depois da avaliação honesta, o modelo selecionado é reajustado em `train + validation` (`X_dev`) com os mesmos hiperparâmetros. O threshold continua sendo o escolhido em validação. A submissão é gerada apenas no fim do notebook.

In [0]:
selected_family = best_candidate['family']
selected_params = best_candidate['params']

# Recria o estimador da mesma família para evitar reaproveitar estado ajustado.
if selected_family == 'Logistic Regression':
    final_estimator = LogisticRegression(solver='liblinear', max_iter=1000, random_state=RANDOM_STATE)
elif selected_family == 'Decision Tree':
    final_estimator = DecisionTreeClassifier(random_state=RANDOM_STATE)
elif selected_family == 'Random Forest':
    final_estimator = RandomForestClassifier(n_estimators=60, n_jobs=1, random_state=RANDOM_STATE)
elif selected_family == 'Extra Trees':
    final_estimator = ExtraTreesClassifier(n_estimators=60, n_jobs=1, random_state=RANDOM_STATE)
else:
    raise ValueError(f'Unexpected selected family: {selected_family}')

final_model = build_pipeline(final_estimator, X_reference=X_dev)
final_model.set_params(**selected_params)
final_model.fit(X_dev, y_dev)

external_proba = final_model.predict_proba(X_external)[:, 1]
external_pred = (external_proba >= selected_threshold).astype(int)

submission = pd.DataFrame({
    ID_COL: test_external[ID_COL],
    TARGET: external_pred,
})

submission_path = 'DM2NT_Group05_Version06_notebook_v9_corrected_vanilla_sklearn.csv'
submission.to_csv(submission_path, index=False)

print('Submission saved to:', SUBMISSION_PATH)
print('External positive prediction rate:', submission[TARGET].mean().round(4))
submission.head()

Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🏃 View run resilient-deer-747 at: https://dbc-701a66cb-661b.cloud.databricks.com/ml/experiments/2760995407200007/runs/669fea301d794ad1937a96980654d221
🧪 View experiment at: https://dbc-701a66cb-661b.cloud.databricks.com/ml/experiments/2760995407200007
Submission saved to: /mnt/data/submission_corrected_vanilla_sklearn.csv
External positive prediction rate: 0.7144


,CONTROL_NUMBER,TARGET_B
0,122653,0
1,184239,1
2,5172,1
3,135377,1
4,62119,1


## 13. Conclusão técnica

Este notebook corrige a organização e as inconsistências da versão anterior. A decisão final é baseada em validação, o teste interno é usado uma única vez e a submissão fica no final.

Sobre a meta de F1 0.45–0.51: o notebook procura automaticamente essa faixa em validação. Caso ela não seja atingida, a conclusão correta é que não há evidência suficiente para garantir essa faixa sem leakage ou ajuste indevido ao teste interno. Nesse caso, o resultado honesto é reportar o melhor modelo validado e discutir as limitações do sinal disponível.